In [ ]:
from importlib import reload
import os
import sys
# Set up paths for utility imports
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')

os.chdir(current_dir)
# Ensure the utilities directory is in the import path
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
    
import plotting
import numpy as np
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
 

In [ ]:
# Load the .npz file
data = np.load("infinite_analytical_solution.npz")

x = data["x"]
y = data["y"]
u = data["u"]

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# Load data
# ============================================================
data = np.load("infinite_analytical_solution.npz")

x = data["x"]
y = data["y"]
u = data["u"]

# Flatten if necessary
x = x.reshape(-1, 1)
y = y.reshape(-1, 1)
u = u.reshape(-1, 1)

X = np.hstack((x, y))

X = torch.tensor(X, dtype=torch.float32)
U = torch.tensor(u, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = X.to(device)
U = U.to(device)

# ============================================================
# MLP
# ============================================================
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 45),
            nn.Atan(),
            nn.Linear(45, 45),
            nn.Atan(),
            nn.Linear(45, 1)
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)

# ============================================================
# Loss
# ============================================================
criterion = nn.MSELoss()

# ============================================================
# Adam
# ============================================================
adam = optim.Adam(
    model.parameters(),
    lr=1e-3
)

epochs_adam = 5000

print("Training with Adam...")

for epoch in range(epochs_adam):

    adam.zero_grad()

    pred = model(X)

    loss = criterion(pred, U)

    loss.backward()

    adam.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d}  Loss = {loss.item():.3e}")

# ============================================================
# LBFGS
# ============================================================
lbfgs = optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=5000,
    tolerance_grad=1e-9,
    tolerance_change=1e-12,
    history_size=100,
    line_search_fn="strong_wolfe"
)

print("\nTraining with L-BFGS...")

def closure():

    lbfgs.zero_grad()

    pred = model(X)

    loss = criterion(pred, U)

    loss.backward()

    return loss

lbfgs.step(closure)

# ============================================================
# Evaluation
# ============================================================
model.eval()

with torch.no_grad():

    pred = model(X)

    mse = criterion(pred, U).item()

    rel_l2 = (
        torch.linalg.norm(pred - U)
        / torch.linalg.norm(U)
    ).item()

print("\nResults")
print("-----------------------")
print(f"MSE Loss      : {mse:.3e}")
print(f"Relative L2   : {rel_l2:.3e}")